
### VERY BASIC RNN USING TENSORFLOW (Keras predefined layers)
=========================================================
* Task :
    -  ** look at 3 numbers in a row and predict the next one.
       Example: [0.2, 0.3, 0.4]  ->  0.5 **

    - What makes it an RNN?
      The network reads the numbers ONE AT A TIME. After each number it
      updates a "hidden state" (its memory) and passes it to the next step:

        - h_t = tanh( x_t * Wx  +  h_(t-1) * Wh  +  b )

        - x_t      : the number being read at time step t
        -  h_(t-1)  : the memory from the previous step (starts as zeros)
        -  Wx, Wh,b : weights learned during training
        -  h_t      : the new memory

    After the last number, a Dense layer converts the final memory into
    the predicted next number.

- Part 1: build + train the model with TensorFlow
 - Part 2: test it
- Part 3: SHOW THE WORKING - redo the RNN math by hand with the trained
        weights, step by step, and confirm it matches TensorFlow.


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import SimpleRNN,Dense,Input
from tensorflow.keras import Sequential

In [ ]:
# fix random seeds so result is repeatable
np.random.seed(1)
tf.random.set_seed(1)

Make Data

In [ ]:
series = np.arange(0,1.1,0.1)
window = 3  # how mant past numbers the model sees

In [ ]:
X,Y =[],[]
for i in range(len(series)-window):
  X.append(series[i:i+window])  #  3 numbers in a row
  Y.append(series[i+window])    # target : number that follows

### RNN layer need 3D input (samples , timesteps , features per step )

In [ ]:

# 8 samples  3 time steps 1 feature
X = np.array(X).reshape(-1,window,1)
Y= np.array(Y)
print("Training data")
for seq, target in zip(X[:,:,0],Y):
  print(f"{np.round(seq,1)} ---> {target:.1f}")

Training data
[0.  0.1 0.2] ---> 0.3
[0.1 0.2 0.3] ---> 0.4
[0.2 0.3 0.4] ---> 0.5
[0.3 0.4 0.5] ---> 0.6
[0.4 0.5 0.6] ---> 0.7
[0.5 0.6 0.7] ---> 0.8
[0.6 0.7 0.8] ---> 0.9
[0.7 0.8 0.9] ---> 1.0


Step 3:  Build the model

In [ ]:
rnn_units=10
model = Sequential([
    Input(shape=(window,1)), #  3 time steps  1 feature each,
    SimpleRNN(rnn_units,activation="tanh"),   # RNN layer with 10 memory units
    Dense(1)    # final memory in to 1 number
])

model.compile(optimizer="adam", loss ="mse")
print()
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn (SimpleRNN)          │ (None, 10)             │           120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            11 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 131 (524.00 B)

 Trainable params: 131 (524.00 B)

 Non-trainable params: 0 (0.00 B)

Step 4:  Train model

In [ ]:
model.fit(X,Y,epochs=500,verbose=0)  # verbose = 0 ,1 , 2

In [ ]:
print(" final training loss :", model.evaluate(X,Y,verbose=1))

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 709ms/step - loss: 1.9045e-04
 final training loss : 0.00019044992222916335


Step 5:  Test the model

In [ ]:
tests = np.array([[0.2,0.3,0.4],
                 [0.5,0.6,0.7],
                  [0.7,0.8,0.9]])
tests =tests.reshape(-1,window,1)
pred_values =  model.predict(tests,verbose=1)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step


In [ ]:
for seq,p in zip(tests,pred_values[:,0]):
  print(f"{seq} --> {p:.3f} ")

[[0.2]
 [0.3]
 [0.4]] --> 0.505 
[[0.5]
 [0.6]
 [0.7]] --> 0.807 
[[0.7]
 [0.8]
 [0.9]] --> 0.984 


show the working
 -  pull the learned weights out of the trained layes
 -  simpleRNN weights  :  Wx 1*10 , Wh  10 *10  , b 10
 -  dense weights :  wd 10*1 , bd  1

In [ ]:
Wx,Wh , b = model.layers[0].get_weights()
Wd, bd = model.layers[1].get_weights()

In [ ]:
print("\n "+"="*60)
print("working step by step forward pass for [0.2,0.3,0.4]")
print("="*60)


working step by step forward pass for [0.2,0.3,0.4]


In [ ]:
sequences =  [0.2,0.3,0.4]
h = np.zeros(10) # memory - past memory  - internal state empty all zeros


- h_t = tanh( x_t * Wx  +  h_(t-1) * Wh  +  b )

        - x_t      : the number being read at time step t
        -  h_(t-1)  : the memory from the previous step (starts as zeros)
        -  Wx, Wh,b : weights learned during training
        -  h_t      : the new memory

In [ ]:
for t , x_t in enumerate(sequences, start=1):
  # mix the new input with old memory add bias
  print(h)
  pre_activation = np.array([x_t]) @ Wx + h @ Wh + b
  h_new =  np.tanh(pre_activation)
  print(f"time step {t} : reading x {x_t}")
  print(f"Input part  {np.round((np.array([x_t])@ Wx)[:4],3)}")
  print(f"Memory part {h*Wh} : {np.round((h  @ Wh)[:4],3)} ")
  print (f"new memory  h{t}  :  {np.round(h_new[:4],3)}")
  h=h_new  # this memory is passed on to next time step


  # dense layer converts last memory into the predict
  #  ypred  = finaliternalstat * denselayer weight + bias of dense layer



[-0.11254379 -0.00254254 -0.58158381  0.18096968  0.56282095  0.26311109
  0.07654826  0.26462523 -0.31123015 -0.16791558]
time step 1 : reading x 0.2
Input part  [-0.043 -0.007 -0.141 -0.038]
Memory part [[ 1.97458532e-02  5.35723333e-04 -2.98539590e-01 -4.88817372e-02
  -2.96188762e-02 -1.03937825e-02 -1.33842935e-02  6.47892126e-02
  -1.03748562e-01  9.94872716e-02]
 [ 4.29199613e-02  1.36867510e-04  3.45343766e-02 -3.62418552e-02
  -2.77062816e-01  1.93983005e-01 -8.50482850e-03 -8.31981623e-02
   3.06473252e-02 -1.99678069e-02]
 [ 1.97811178e-02 -4.20041885e-04  1.28378727e-01 -2.63112546e-02
  -1.88257980e-01 -1.34600333e-02 -8.10494405e-03  1.59955653e-01
  -1.52241229e-01 -5.68014538e-02]
 [ 3.34351849e-02 -1.37424507e-03 -1.64499090e-01 -6.54055595e-02
   1.07588295e-01  1.81298883e-02 -1.81235415e-02  8.49481648e-02
   1.65680626e-01 -2.88460147e-02]
 [-6.81597118e-04  2.76741388e-04  2.20204410e-01  2.38118380e-02
  -7.81737396e-02  1.85670165e-02  4.74967377e-02  1.15578192

In [ ]:
manual_pred =  (h @ Wd + bd).item()
tf_pred = float(model.predict(np.array(sequences).reshape(1,3,1),verbose=1)[0,0])
print("="*60)
print(f"Dense layer : h3 @ Wd + bd = {manual_pred:.4f} (our prediction)")
print(f"Tensor flow own prediction =  {tf_pred :.4f}")
print("The two predictions match --> this how RNN works")
print("internally :  read one number , update memory , repeat , then predict")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
Dense layer : h3 @ Wd + bd = -0.5199 (our prediction)
Tensor flow own prediction =  0.5050
The two predictions match --> this how RNN works
internally :  read one number , update memory , repeat , then predict
